# Phase 1: Train Sentiment Analysis Model

In this notebook, we fine-tune `XLM-RoBERTa` for multilingual multi-class sentiment analysis.

**Objective**:
- Load the preprocessed sentiment dataset (Positive, Negative, Neutral)
- Initialize `XLMRobertaForSequenceClassification` with 3 labels
- Train the model using PyTorch
- Evaluate using Accuracy
- Save the final weights to `models/sentiment_model`

In [ ]:
import os
import torch
import pandas as pd
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score

MODEL_NAME = 'xlm-roberta-base'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. Dataset Definition
We wrap our loaded tokenized data into a standard PyTorch `Dataset`. Labels should be mapped from Negative=0, Neutral=1, Positive=2 (or whatever mapping the raw data uses).

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

## 2. Model Initialization & Training Loop

In [ ]:
def train_model(train_loader, val_loader, epochs=3, lr=2e-5):
    # Note num_labels=3 for Positive, Neutral, Negative
    model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    model.to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            
        avg_train_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} | Average Training Loss: {avg_train_loss:.4f}")
        
        # Validation Phase
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1)
                
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                
        acc = accuracy_score(val_labels, val_preds)
        print(f"Validation Accuracy: {acc:.4f}")
        
    return model


## 3. Save Model
Once training hits our Accuracy > 80% target, we save the model for inference.

In [ ]:
def save_model(model, tokenizer, path="../models/sentiment_model"):
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    print(f"Model saved to {path}")

# tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)
# save_model(model, tokenizer)